<a href="https://colab.research.google.com/github/JaimRM/QuantitativeFinance/blob/main/QoE_model_Acciona.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install yfinance pandas

In [ ]:
import pandas as pd
import yfinance as yf


def quality_of_earnings_model(ticker_symbol):
    print(f"=== Extrayendo datos financieros para {ticker_symbol} ===")
    company = yf.Ticker(ticker_symbol)

    # 1. Obtener estados financieros anuales
    income_stmt = company.financials
    cash_flow = company.cashflow
    balance_sheet = company.balance_sheet

    if income_stmt.empty or cash_flow.empty or balance_sheet.empty:
        raise ValueError(
            "No se pudieron descargar los datos completos. Verifica el ticker."
        )

    # Transponer para trabajar con los años en las filas
    df_inc = income_stmt.T
    df_cf = cash_flow.T
    df_bs = balance_sheet.T

    # 2. Extraer las variables clave del modelo
    # Nota: yfinance mapea las etiquetas en inglés
    qoe_data = pd.DataFrame(index=df_inc.index)

    # Beneficio Neto e Ingresos
    qoe_data["Net Income"] = df_inc["Net Income"]
    qoe_data["Total Revenue"] = df_inc["Total Revenue"]

    # Flujos de Caja
    qoe_data["Operating Cash Flow"] = df_cf["Operating Cash Flow"]
    if "Investing Cash Flow" in df_cf.columns:
        qoe_data["Investing Cash Flow"] = df_cf["Investing Cash Flow"]
    else:
        qoe_data["Investing Cash Flow"] = 0

    # Balance General
    qoe_data["Total Assets"] = df_bs["Total Assets"]

    # Cuentas por cobrar (Accounts Receivable) si están disponibles
    if "Receivables" in df_bs.columns:
        qoe_data["Receivables"] = df_bs["Receivables"]
    elif "Accounts Receivable" in df_bs.columns:
        qoe_data["Receivables"] = df_bs["Accounts Receivable"]
    else:
        qoe_data["Receivables"] = None

    # 3. Calcular métricas de Calidad de Beneficios (QoE)

    # A. Ratio de Conversión de Caja (Cash Conversion Ratio)
    # Lo ideal es que sea > 1 (Más caja que beneficio contable)
    qoe_data["CFO / Net Income"] = (
        qoe_data["Operating Cash Flow"] / qoe_data["Net Income"]
    )

    # B. Ratio de Devengo de Sloan (Sloan Accrual Ratio)
    # Fórmula: (Net Income - CFO - CFI) / Total Assets
    # Interpretación: Entre -10% y 10% es óptimo. > 10% indica ganancias artificiales/contables.
    qoe_data["Sloan Accrual Ratio"] = (
        qoe_data["Net Income"]
        - qoe_data["Operating Cash Flow"]
        - qoe_data["Investing Cash Flow"]
    ) / qoe_data["Total Assets"]

    # C. Crecimiento de Ventas vs Cuentas por Cobrar
    if qoe_data["Receivables"].notnull().all():
        qoe_data["Rev_Growth"] = qoe_data["Total Revenue"].pct_change(periods=-1)
        qoe_data["Rec_Growth"] = qoe_data["Receivables"].pct_change(periods=-1)
        # Si las cuentas por cobrar crecen mucho más rápido que las ventas, bandera roja.
        qoe_data["Receivables_vs_Revenue_Gap"] = (
            qoe_data["Rec_Growth"] - qoe_data["Rev_Growth"]
        )

    # Ordenar cronológicamente (de más antiguo a más reciente)
    qoe_data = qoe_data.sort_index()

    return qoe_data


# Ejecutar el modelo para Acciona
try:
    qoe_results = quality_of_earnings_model("ANA.MC")

    # Mostrar columnas principales del análisis
    columns_to_show = [
        "Total Revenue",
        "Net Income",
        "Operating Cash Flow",
        "CFO / Net Income",
        "Sloan Accrual Ratio",
    ]

    if "Receivables_vs_Revenue_Gap" in qoe_results.columns:
        columns_to_show.append("Receivables_vs_Revenue_Gap")

    print("\n=== RESULTADOS DEL MODELO QUALITY OF EARNINGS ===")
    print(qoe_results[columns_to_show].round(4).to_string())

except Exception as e:
    print(f"Error al ejecutar el modelo: {e}")

=== Extrayendo datos financieros para ANA.MC ===

=== RESULTADOS DEL MODELO QUALITY OF EARNINGS ===
            Total Revenue   Net Income  Operating Cash Flow  CFO / Net Income  Sloan Accrual Ratio
2021-12-31            NaN          NaN                  NaN               NaN                  NaN
2022-12-31   1.119500e+10  441000000.0         1.648000e+09            3.7370               0.0326
2023-12-31   1.702100e+10  541000000.0         1.695000e+09            3.1331               0.0649
2024-12-31   1.919000e+10  422000000.0         2.239000e+09            5.3057               0.0187
2025-12-31   2.023600e+10  803000000.0         2.148000e+09            2.6750              -0.0282
